# DICA Tutorial: Decision-Informed Conformal Adaptation

**What you'll learn:**
1. What problem DICA solves
2. How to set up a predict-then-optimize pipeline
3. How to run DICA vs standard conformal (UCA)
4. How to interpret the output
5. How to tune parameters

**Prerequisites:** `pip install conformal-ops matplotlib`

---

## 1. The Problem

You're an ICU operations manager. Each shift, you allocate a fixed nursing budget across 20 patients based on predicted acuity (length of stay). You solve a linear program (LP):

$$\min_{z} \hat{c}^\top z \quad \text{s.t.} \quad \sum_j z_j = B, \quad 0.1 \leq z_j \leq 1$$

But your predictions $\hat{c}$ have errors. **Conformal prediction** builds uncertainty margins $r_j$ around each prediction, and you solve a **robust LP** with $\hat{c} + r$ instead. This gives you coverage (90% of the time, your margins contain the truth) but costs more — the **Price of Coverage (PoC)**.

**DICA's insight:** Standard conformal gives every patient the same margin. But the LP assigns most patients to minimum staffing — those margins are wasted. DICA redistributes: tighter margins on low-allocation patients, wider on high-allocation patients.

## 2. Setup: Define the LP and Generate Data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from conformal_ops import DICA

# --- LP setup: Nurse staffing ---
d = 20           # 20 patients per shift
budget = 12.0    # total nursing budget
A_eq = np.ones((1, d))
b_eq = np.array([budget])
bounds = [(0.1, 1.0)] * d  # min 0.1, max 1.0 per patient

print(f"Problem: allocate budget={budget} across {d} patients")
print(f"Each patient gets between 0.1 and 1.0 nursing units")
print(f"LP: min c^T z  s.t. sum(z) = {budget}, 0.1 ≤ z_j ≤ 1.0")

In [ ]:
# --- Generate synthetic patient data ---
# Simulate 500 online rounds (shifts)
# Distribution shift at round 300: mean LOS increases (e.g., flu season)

T = 500
shift_at = 300
rng = np.random.RandomState(42)

data = []
for t in range(T):
    mean_los = 2.0 if t < shift_at else 4.0  # shift!
    los_true = rng.exponential(mean_los, size=d)
    los_pred = los_true + rng.normal(0, 0.8, size=d)  # noisy predictions
    
    # Convert LOS to cost vectors (higher LOS = higher cost)
    c_true = 0.5 + np.maximum(los_true, 0) / 10
    c_pred = 0.5 + np.maximum(los_pred, 0) / 10
    data.append((c_pred, c_true))

print(f"Generated {T} rounds of synthetic data")
print(f"Distribution shift at round {shift_at} (mean LOS: 2.0 → 4.0)")
print(f"Example c_pred (round 0): [{data[0][0][:5].round(3)}...]")
print(f"Example c_true (round 0): [{data[0][1][:5].round(3)}...]")

## 3. Run DICA vs UCA (Standard Conformal)

**Key parameters:**
| Parameter | Default | What it does |
|-----------|---------|--------------|
| `alpha` | 0.10 | Target miscoverage (0.10 = 90% coverage) |
| `beta` | 0.5 | Redistribution strength (0 = UCA, 1 = fully proportional) |
| `eta` | 0.05 | Gibbs-Candès step size (how fast alpha adapts) |
| `window` | 150 | How many recent scores to keep for quantile estimation |

In [ ]:
# --- Run both methods on the same data ---
results = {}

for name, beta in [("UCA (standard)", 0.0), ("DICA (β=0.5)", 0.5)]:
    model = DICA(alpha=0.10, beta=beta)
    rounds = []
    
    for t, (c_pred, c_true) in enumerate(data):
        # One DICA step: get radii → solve robust LP → observe truth → update
        r = model.step(c_pred, c_true, A_eq=A_eq, b_eq=b_eq, bounds=bounds)
        rounds.append(r)
    
    results[name] = {
        "rounds": rounds,
        "stats": model.get_results(),
        "poc_history": model.poc_history,
        "model": model,
    }

# --- Print summary ---
print(f"{'Method':20s}  {'PoC':>8s}  {'Coverage':>10s}  {'DICA-cov':>10s}")
print("-" * 55)
for name, res in results.items():
    s = res["stats"]
    print(f"{name:20s}  {s['avg_poc']:+7.2%}  {s['coverage']:>9.1%}  {s['dica_coverage']:>9.1%}")

## 4. Visualize: Coverage Trajectory

Both methods should track ~90% coverage, even through the distribution shift at round 300. The Gibbs-Candès update adapts automatically.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
w = 50  # rolling window

for name, color, ls in [("UCA (standard)", "gray", "--"), ("DICA (β=0.5)", "#2196F3", "-")]:
    covs = [r["std_covered"] for r in results[name]["rounds"]]
    rolling = np.convolve(covs, np.ones(w)/w, mode="valid")
    ax.plot(range(w-1, T), rolling, label=name, color=color, linewidth=2, linestyle=ls)

ax.axhline(0.9, color="red", linestyle=":", alpha=0.6, label="Target (90%)")
ax.axvline(shift_at, color="orange", linestyle=":", alpha=0.6, label="Distribution shift")
ax.set_xlabel("Online Round (shift)", fontsize=12)
ax.set_ylabel("Rolling Coverage (w=50)", fontsize=12)
ax.set_title("Coverage Trajectory: Both Methods Track 90%", fontsize=14)
ax.legend(fontsize=11)
ax.set_ylim(0.6, 1.05)
fig.tight_layout()
plt.show()

## 5. Visualize: Price of Coverage (PoC)

DICA should have consistently lower PoC than UCA — same coverage, lower cost.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

for name, color, ls in [("UCA (standard)", "gray", "--"), ("DICA (β=0.5)", "#2196F3", "-")]:
    pocs = results[name]["poc_history"]
    rolling = np.convolve(pocs, np.ones(w)/w, mode="valid")
    ax.plot(range(w-1, T), rolling * 100, label=name, color=color, linewidth=2, linestyle=ls)

ax.axvline(shift_at, color="orange", linestyle=":", alpha=0.6, label="Distribution shift")
ax.set_xlabel("Online Round (shift)", fontsize=12)
ax.set_ylabel("Price of Coverage (%)", fontsize=12)
ax.set_title("PoC Trajectory: DICA Reduces the Cost of Coverage", fontsize=14)
ax.legend(fontsize=11)
fig.tight_layout()
plt.show()

# Print the savings
uca_poc = results["UCA (standard)"]["stats"]["avg_poc"]
dica_poc = results["DICA (β=0.5)"]["stats"]["avg_poc"]
print(f"UCA average PoC:  {uca_poc:+.2%}")
print(f"DICA average PoC: {dica_poc:+.2%}")
if uca_poc > 0:
    print(f"Relative reduction: {(1 - dica_poc/uca_poc):.0%}")

## 6. Visualize: How DICA Reshapes Radii

This is the core mechanism. Look at how DICA tightens radii on low-allocation patients and widens on high-allocation patients.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
x = np.arange(d)

# Panel (a): Radii comparison
ax = axes[0]
r_uca = results["UCA (standard)"]["rounds"][-1]["radii"]
r_dica = results["DICA (β=0.5)"]["rounds"][-1]["radii"]
ax.bar(x - 0.2, r_uca, 0.4, label="UCA (uniform)", color="gray", alpha=0.7)
ax.bar(x + 0.2, r_dica, 0.4, label="DICA (reshaped)", color="#2196F3", alpha=0.7)
ax.set_xlabel("Patient j")
ax.set_ylabel("Conformal radius r_j")
ax.set_title("(a) Radii: UCA vs DICA")
ax.legend(fontsize=9)

# Panel (b): Allocation EMA
ax = axes[1]
alloc = results["DICA (β=0.5)"]["model"].allocation_ema
colors = ["#2196F3" if a > np.median(alloc) else "lightgray" for a in alloc]
ax.bar(x, alloc, color=colors)
ax.set_xlabel("Patient j")
ax.set_ylabel("Allocation EMA z̄_j")
ax.set_title("(b) LP Allocation Feedback")
ax.axhline(np.median(alloc), color="red", linestyle=":", alpha=0.5, label="Median")
ax.legend(fontsize=9)

# Panel (c): LP solution (last round)
ax = axes[2]
z_opt = results["DICA (β=0.5)"]["rounds"][-1]["z_opt"]
colors_z = ["#2196F3" if z > 0.15 else "lightgray" for z in z_opt]
ax.bar(x, z_opt, color=colors_z)
ax.axhline(0.1, color="red", linestyle=":", alpha=0.5, label="Lower bound (0.1)")
ax.set_xlabel("Patient j")
ax.set_ylabel("Allocation z*_j")
ax.set_title("(c) LP Solution (last round)")
ax.legend(fontsize=9)

fig.suptitle("How DICA Works: Allocation Feedback → Radii Redistribution", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

# Count sparsity
n_at_lb = np.sum(z_opt < 0.15)
print(f"\nLP sparsity: {n_at_lb}/{d} patients at lower bound ({n_at_lb/d:.0%})")
print(f"DICA saves cost on these {n_at_lb} patients by tightening their radii")

## 7. Tuning β: The Redistribution Strength

`beta` controls how aggressively DICA reshapes radii. Let's sweep it to find the sweet spot.

In [ ]:
betas = [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
sweep_results = []

for beta in betas:
    model = DICA(alpha=0.10, beta=beta)
    for c_pred, c_true in data:
        model.step(c_pred, c_true, A_eq=A_eq, b_eq=b_eq, bounds=bounds)
    s = model.get_results()
    sweep_results.append(s)
    
# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# PoC vs beta
ax = axes[0]
pocs = [s["avg_poc"] * 100 for s in sweep_results]
ax.plot(betas, pocs, "o-", color="#2196F3", linewidth=2, markersize=8)
ax.set_xlabel("β (redistribution strength)", fontsize=12)
ax.set_ylabel("Average PoC (%)", fontsize=12)
ax.set_title("PoC vs β (U-shaped)", fontsize=13)
ax.axvline(0.5, color="red", linestyle=":", alpha=0.5, label="Default β=0.5")
ax.legend()

# Coverage vs beta
ax = axes[1]
covs = [s["coverage"] * 100 for s in sweep_results]
dica_covs = [s["dica_coverage"] * 100 for s in sweep_results]
ax.plot(betas, covs, "s-", color="gray", linewidth=2, markersize=8, label="Scalar coverage")
ax.plot(betas, dica_covs, "o-", color="#2196F3", linewidth=2, markersize=8, label="DICA-radii coverage")
ax.axhline(90, color="red", linestyle=":", alpha=0.5)
ax.set_xlabel("β (redistribution strength)", fontsize=12)
ax.set_ylabel("Coverage (%)", fontsize=12)
ax.set_title("Coverage vs β", fontsize=13)
ax.legend()

fig.tight_layout()
plt.show()

# Print table
print(f"{'β':>5s}  {'PoC':>7s}  {'Scalar Cov':>10s}  {'DICA Cov':>10s}")
print("-" * 38)
for beta, s in zip(betas, sweep_results):
    print(f"{beta:5.1f}  {s['avg_poc']:+6.2%}  {s['coverage']:>9.1%}  {s['dica_coverage']:>9.1%}")

## 8. Inspect a Single Round

Let's look at what happens in one round to understand the full pipeline.

In [ ]:
# Pick round 400 (after distribution shift, after warmup)
r = results["DICA (β=0.5)"]["rounds"][400]

print("=== Round 400 (DICA) ===")
print(f"LP solution z*:     [{r['z_opt'][:5].round(3)}...]")
print(f"Radii used:         [{r['radii'][:5].round(4)}...]")
print(f"True cost:          {r['cost']:.4f}")
print(f"Nominal cost:       {r['nominal_cost']:.4f}")
print(f"PoC this round:     {r['poc']:+.2%}")
print(f"Scalar covered:     {'YES' if r['std_covered'] else 'NO'}")
print(f"DICA-radii covered: {'YES' if r['dica_covered'] else 'NO'}")
print()

# Show which patients are at lower bound vs high allocation
z = r["z_opt"]
at_lb = np.sum(z < 0.15)
at_ub = np.sum(z > 0.95)
print(f"Patients at lower bound (z ≈ 0.1): {at_lb}/{d}")
print(f"Patients at upper bound (z ≈ 1.0): {at_ub}/{d}")
print(f"Patients fractional:               {d - at_lb - at_ub}/{d}")

## 9. Use DICA with Your Own LP

DICA works with any LP. Here's the minimal interface:

```python
from conformal_ops import DICA

# 1. Define your LP constraints
A_eq, b_eq = ...   # equality constraints (optional)
A_ub, b_ub = ...   # inequality constraints (optional)
bounds = [...]      # variable bounds

# 2. Create DICA
dica = DICA(alpha=0.10, beta=0.5)

# 3. Online loop
for t in range(T):
    c_pred = your_predictor(x_t)   # your predictions
    c_true = observe()              # truth revealed after decision
    
    result = dica.step(c_pred, c_true,
                       A_eq=A_eq, b_eq=b_eq,
                       A_ub=A_ub, b_ub=b_ub,
                       bounds=bounds)
    
    z_opt = result["z_opt"]  # use this for your decision
```

**That's it.** DICA handles all the conformal prediction, radii reshaping, and LP solving internally.

---

## Summary

| What | How |
|------|-----|
| **Install** | `pip install conformal-ops` |
| **Create** | `dica = DICA(alpha=0.10, beta=0.5)` |
| **Run** | `result = dica.step(c_pred, c_true, A_eq=..., b_eq=..., bounds=...)` |
| **Results** | `dica.get_results()` → coverage, PoC, cost |
| **Tune** | Sweep `beta` from 0 to 1, look for U-shaped PoC curve |

**Paper:** Dronavajjala (2026). *Decision-Informed Online Conformal Prediction for ICU Resource Allocation.* PMLR 340, MLHC 2026.